# Feature Engineering: análisis del impacto de variables — predicción de `opinion`

Obligatorio MLSI 2026 S2. Esta notebook contiene:

1. Carga de datos
2. Preprocesamiento (código común en `obl_utils.py`)
3. Diseño del análisis de ablación
4. Resultados de la ablación
5. Comparación pareada por fold
6. Conclusiones

La ingeniería de variables (limpieza, variables nuevas, codificación de categóricas y descarte de columnas) ya está implementada en `obl_utils.py` y es la que usan todas las notebooks de modelos. Acá no se cambia nada de eso: se mide **cuánto aporta cada decisión**, sacando o modificando una por vez y comparando el AUC con la misma validación cruzada que el resto de los modelos.

## 1. Carga de datos

In [1]:
import os
import urllib.request

# El código común (preprocesamiento y evaluación) está en obl_utils.py.
# En Colab no está el archivo, así que se descarga del repo.
if not os.path.exists('obl_utils.py'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/juanmareolon/OBL_ML/main/obl_utils.py', 'obl_utils.py')

import warnings
import numpy as np
import pandas as pd
from obl_utils import *

warnings.filterwarnings('ignore', category=FutureWarning)

CARPETA_RESULTADOS = 'resultados/Feature_Engineering'
os.makedirs(CARPETA_RESULTADOS, exist_ok=True)

train, test = cargar_datos()
X, y, X_test, ids_test = preparar_datos(train, test)
print(X.shape)

(3602, 15)


## 2. Preprocesamiento

Se usa sin cambios el de `obl_utils.py` (detalle en `Arbol_Clasificacion.ipynb`, sección 2):

- **Variables creadas:** `log_roi` (revenue / budget), `anio` y `mes` (de `release_date`), `sin_keywords` y `cant_keywords`.
- **Indicadores de dato desconocido:** `budget_desconocido` y `revenue_desconocido` (montos 0 o menores a 1000).
- **Transformación log(1 + x):** `budget`, `revenue`, ROI, `vote_count` y `popularity`.
- **Categóricas:** one-hot de `main_genre_top10` (11 valores) y de `original_language`, agrupando los idiomas con menos de 20 películas en una categoría "otros" (`min_frequency=20`).
- **`director`:** casi 2000 valores distintos; por defecto no se usa. `crear_preprocesador(director=...)` permite agregarlo con frequency encoding o con frequency + target encoding.
- **Descartadas:** `status`, `original_title` y `overview`.

Algunos números que justifican la selección de variables:

In [2]:
resumen = pd.DataFrame({
    'valores distintos': train[['status', 'original_title', 'director', 'original_language', 'main_genre_top10']].nunique(),
})
resumen['% del valor más frecuente'] = [
    round(100 * train[c].value_counts(normalize=True).iloc[0], 1) for c in resumen.index]
print('Filas del train:', len(train))
resumen

Filas del train: 3602


,valores distintos,% del valor más frecuente
status,3,99.8
original_title,3600,0.1
director,1942,0.7
original_language,37,93.7
main_genre_top10,11,24.8


In [3]:
idiomas = train['original_language'].value_counts()
print('Idiomas con 20 o más películas:', list(idiomas[idiomas >= 20].index))
print('Películas que quedan en "otros":', idiomas[idiomas < 20].sum(), 'de', len(train))
directores = train['director'].value_counts()
print('Directores con una sola película:', (directores == 1).sum(), 'de', len(directores))

Idiomas con 20 o más películas: ['en', 'fr', 'es']
Películas que quedan en "otros": 146 de 3602
Directores con una sola película: 1275 de 1942


## 3. Diseño del análisis de ablación

Se parte del preprocesamiento completo (**base**) y en cada variante se **quita o modifica un solo grupo de variables**, dejando todo lo demás igual. Cada variante se evalúa con los mismos 5 folds estratificados (`crear_cv()`), con el preprocesador dentro del pipeline, igual que en el resto de las notebooks.

Se usan dos modelos representativos y rápidos:

- **Regresión logística** (C = 1, L2, el modelo elegido en `Regresion_Logistica.ipynb`): es lineal, así que es sensible a la escala y a la forma de las variables.
- **Gradient Boosting** (profundidad 3, 300 árboles, learning rate 0,05, subsample 0,8, el elegido en `Boosting.ipynb`): basado en árboles, invariante a transformaciones monótonas como el logaritmo.

Para quitar variables se reutiliza `crear_preprocesador()` y solo se filtran las columnas de cada transformador. La variante sin logaritmo deshace la transformación con `expm1` antes de entrar al pipeline.

In [4]:
from scipy import stats
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score


def preprocesador_variante(excluir=(), director='ninguno', agrupar_idiomas=True):
    """El preprocesador de obl_utils, sin las columnas de `excluir`.
    Con agrupar_idiomas=False cada idioma tiene su propia columna (sin 'otros')."""
    prep = crear_preprocesador(director)
    transformadores = []
    for nombre, trans, cols in prep.transformers:
        cols = [c for c in cols if c not in excluir]
        if not cols:
            continue
        if nombre == 'idioma' and not agrupar_idiomas:
            trans = clone(trans).set_params(min_frequency=None, handle_unknown='ignore')
        transformadores.append((nombre, trans, cols))
    prep.transformers = transformadores
    return prep


def ttest_corregido(dif, n_folds=N_FOLDS):
    """t-test pareado corregido de Nadeau y Bengio (2003), el mismo de Stacking.ipynb:
    agranda la varianza porque los folds comparten datos de entrenamiento."""
    k = len(dif)
    var = np.var(dif, ddof=1) * (1 / k + 1 / (n_folds - 1))
    t = dif.mean() / np.sqrt(var)
    return 2 * stats.t.sf(abs(t), df=k - 1)


def auc_por_fold(modelo, Xv, **kw):
    pipe = Pipeline([('prep', preprocesador_variante(**kw)), ('modelo', modelo)])
    return cross_val_score(pipe, Xv, y, cv=crear_cv(), scoring='roc_auc', n_jobs=-1)


MODELOS = {
    'Logística': LogisticRegression(random_state=SEED),
    'Gradient Boosting': GradientBoostingClassifier(max_depth=3, n_estimators=300, learning_rate=0.05,
                                                    subsample=0.8, random_state=SEED),
}

# Variante sin logaritmo: se vuelve a la escala original
X_sin_log = X.copy()
for c in ['log_budget', 'log_revenue', 'log_roi', 'log_vote_count', 'log_popularity']:
    X_sin_log[c] = np.expm1(X[c])

CREADAS = ['log_roi', 'anio', 'mes', 'sin_keywords', 'cant_keywords']
VARIANTES = [
    ('Base (preprocesamiento completo)', X, {}),
    ('Sin ROI', X, dict(excluir=['log_roi'])),
    ('Sin año y mes', X, dict(excluir=['anio', 'mes'])),
    ('Sin variables de keywords', X, dict(excluir=['sin_keywords', 'cant_keywords'])),
    ('Sin ninguna variable creada', X, dict(excluir=CREADAS)),
    ('Sin indicadores de desconocido', X, dict(excluir=['budget_desconocido', 'revenue_desconocido'])),
    ('Sin transformación log', X_sin_log, {}),
    ('Idiomas sin agrupar', X, dict(agrupar_idiomas=False)),
    ('Sin idioma', X, dict(excluir=['original_language'])),
    ('Sin vote_count', X, dict(excluir=['log_vote_count'])),
    ('Con director (frequency)', X, dict(director='freq')),
    ('Con director (frequency + target)', X, dict(director='freq+target')),
]

## 4. Resultados de la ablación

In [5]:
scores = {}   # (variante, modelo) -> AUC de cada fold
for nombre, Xv, kw in VARIANTES:
    for m, modelo in MODELOS.items():
        scores[(nombre, m)] = auc_por_fold(modelo, Xv, **kw)

base = VARIANTES[0][0]
filas = []
for nombre, _, _ in VARIANTES:
    fila = {'variante': nombre}
    for m in MODELOS:
        s = scores[(nombre, m)]
        dif = s - scores[(base, m)]
        fila[f'AUC {m}'] = s.mean()
        fila[f'desvío {m}'] = s.std()
        fila[f'Δ {m}'] = dif.mean()
        fila[f'p {m}'] = np.nan if nombre == base else ttest_corregido(dif)
        fila[f'folds que mejoran {m}'] = np.nan if nombre == base else int((dif > 0).sum())
    filas.append(fila)

ablacion = pd.DataFrame(filas).set_index('variante')
ablacion.to_csv(f'{CARPETA_RESULTADOS}/ablacion.csv')
ablacion[[c for c in ablacion.columns if c.startswith(('AUC', 'Δ'))]].round(4)

,AUC Logística,Δ Logística,AUC Gradient Boosting,Δ Gradient Boosting
variante,,,,
Base (preprocesamiento completo),0.8155,0.0000,0.8358,0.0000
Sin ROI,0.8149,-0.0006,0.8373,0.0015
Sin año y mes,0.8061,-0.0094,0.8223,-0.0135
Sin variables de keywords,0.8154,-0.0001,0.8373,0.0015
Sin ninguna variable creada,0.8040,-0.0114,0.8218,-0.0140
Sin indicadores de desconocido,0.8046,-0.0109,0.8357,-0.0001
Sin transformación log,0.8172,0.0017,0.8358,0.0000
Idiomas sin agrupar,0.8151,-0.0004,0.8365,0.0007
Sin idioma,0.8113,-0.0042,0.8341,-0.0017


## 5. Comparación pareada por fold

Las diferencias entre variantes son del mismo orden que el desvío entre folds (≈ 0,01), así que comparar promedios no alcanza. Como todas las variantes usan **los mismos folds**, se comparan fold a fold contra la base con el **t-test pareado corregido de Nadeau y Bengio (2003)**, el mismo usado en `Stacking.ipynb` (el t-test común es demasiado optimista porque los folds comparten datos de entrenamiento): se cuenta en cuántos de los 5 folds la variante mejora y el p-valor de la diferencia. Con solo 5 folds el test es orientativo; se toma como efecto claro cuando p < 0,05 y la diferencia va en el mismo sentido en (casi) todos los folds.

In [6]:
ablacion[[c for c in ablacion.columns if c.startswith(('Δ', 'p ', 'folds'))]].round(3)

,Δ Logística,p Logística,folds que mejoran Logística,Δ Gradient Boosting,p Gradient Boosting,folds que mejoran Gradient Boosting
variante,,,,,,
Base (preprocesamiento completo),0.000,NaN,NaN,0.000,NaN,NaN
Sin ROI,-0.001,0.027,0.0,0.002,0.546,4.0
Sin año y mes,-0.009,0.020,0.0,-0.014,0.035,0.0
Sin variables de keywords,-0.000,0.943,2.0,0.001,0.356,3.0
Sin ninguna variable creada,-0.011,0.016,0.0,-0.014,0.033,0.0
Sin indicadores de desconocido,-0.011,0.039,0.0,-0.000,0.962,2.0
Sin transformación log,0.002,0.762,3.0,0.000,0.991,2.0
Idiomas sin agrupar,-0.000,0.667,3.0,0.001,0.695,3.0
Sin idioma,-0.004,0.105,1.0,-0.002,0.445,1.0


## 6. Conclusiones

- **Variables creadas.** En conjunto aportan: quitarlas baja el AUC en los dos modelos y en los 5 folds (≈ −0,011 en la logística y −0,014 en Gradient Boosting). Casi todo el aporte viene de **año y mes**; el ROI y las variables de keywords por separado no cambian el resultado de forma apreciable (su información ya está en budget, revenue y en el resto de las variables).
- **Indicadores de desconocido.** Importantes para la **logística** (−0,011 al quitarlos, 5 de 5 folds) y neutros para Gradient Boosting: un modelo lineal no puede separar "dato faltante imputado con la mediana" de "valor mediano" sin el indicador, mientras que los árboles combinan la información de otras variables.
- **Transformación log.** No cambia el AUC en ninguno de los dos modelos. En Gradient Boosting es esperable (los árboles solo usan el orden de los valores); en la logística, el escalado y la regularización ya acotan el efecto de los extremos. Se mantiene porque estabiliza la escala y facilita leer los coeficientes.
- **Idioma.** Agrupar los idiomas poco frecuentes en "otros" no pierde desempeño frente a una columna por idioma (34 de 37 idiomas tienen menos de 20 películas) y evita columnas casi vacías. Quitar el idioma por completo baja apenas el AUC (−0,004 en la logística, no significativo).
- **vote_count** es la variable individual con más impacto: quitarla cuesta ≈ −0,015 en ambos modelos.
- **director.** Con solo frequency encoding no aporta. Con **frequency + target encoding** mejora la logística (+0,007, 5 de 5 folds, p ≈ 0,006) y, sin significancia, Gradient Boosting (+0,005, 4 de 5 folds, p ≈ 0,28). En el árbol de clasificación no había mejorado, y por eso los modelos del informe se entrenaron sin director; queda como mejora posible.